In [1]:
from transformers import AutoTokenizer
import json
from datasets import Dataset
import numpy as np
# from skmultilearn.model_selection import iterative_train_test_split
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

/home/lm2445/.conda/envs/finben_linhai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the trained BERTopic model folder
topic_model = BERTopic.load("PPCBERTopic/bertopic/my_bertopic_model_HDBSCAN")

In [3]:
file_path1 = "stratified_test_data_original.json"  # Update with your actual file path
with open(file_path1, "r") as f:
    data1 = json.load(f)
file_path2 = "stratified_train_data_original.json"  # Update with your actual file path
with open(file_path2, "r") as f:
    data2 = json.load(f)

In [13]:
len(data1 + data2)

1144

In [5]:
# add author feature
# load the raw data and get the message to author dict
file_path1 = "data/message_dataset_0505.jsonl"  # Update with your actual file path
with open(file_path1, "r") as f:
    data_raw1 = [json.loads(line) for line in f]
file_path2 = "data/message_trainset_0617_2.jsonl"  # Update with your actual file path
with open(file_path2, "r") as f:
    data_raw2 = [json.loads(line) for line in f]
raw_data = data_raw1 + data_raw2
def my_clean(txt):
    txt = txt.strip()
    txt = " ".join(txt.split())
    return txt
transform = {}
for item in raw_data:
    from_whom = "Provider" if item["meta"]["TO_PAT_YN"] == "Y" else "Patient"
    sentence = my_clean(item["context"])
    new_input_text = f"From {from_whom}: {sentence}"
    transform[sentence] = new_input_text

In [8]:
# add author feature
def add_pinfo(data):
    ret = []
    for item in data:
        txt = my_clean(item["text"])
        new_txt = transform[txt]
        ret.append({"text": new_txt, "labels": item["labels"]})
    return ret
new_data1 = add_pinfo(data1)
new_data2 = add_pinfo(data2)     

In [12]:
len(new_data1 + new_data2)

1144

In [18]:
# just test
try_txt = [data1[2]["text"], data1[1]["text"]]
# Example new sentence
new_sentence = try_txt
print (new_sentence)
# Transform it with your trained topic model
new_topics, new_probs = topic_model.transform(new_sentence)

# Get the assigned topic ID
for i in range(len(new_topics)):
    assigned_topic = new_topics[i]
    print(f"Assigned topic: {assigned_topic}")
    
    # Get the top 3 words for this topic
    if assigned_topic != -1:
        topic_words = topic_model.get_topic(assigned_topic)
        top_3_words = [word for word, _ in topic_words[:3]]
        print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
    else:
        print("This sentence was classified as outlier topic (-1). No top words.")


[' Hi, We need to make an office visit for Person1 to review PET scan with Dr. Person2, please let us know what works. Thank you Person3', " Hello Person1! I just set up an appt for me to see Dr. Person2 on Mon. MM/DD/YYYY, 4:30pm. Can Person3 and I make that a combo appt? Or, for Person3, if it's to discuss the test results he / we (Person3 doesn't listen to test results - I do) can do a telephone appt with Dr. Person4 So I am correct to hold off appt with dr. Person5 until PET results are in, yes? THANK YOU for your help! Person6 (for Person3)"]
Assigned topic: 5
Top 3 words for topic 5: ['pet', 'scan', 'cat']
Assigned topic: 110
Top 3 words for topic 110: ['appt', 'appts', 'stein']


In [19]:
type (new_sentence)

list

In [21]:
# still a try
def get_topics(sentences):
    assert type(sentences) == list
    # Transform it with your trained topic model
    new_topics, new_probs = topic_model.transform(sentences)
    # Get the assigned topic ID
    outlier_count = 0
    ret = []
    for i in range(len(new_topics)):
        assigned_topic = new_topics[i]
        #print(f"Assigned topic: {assigned_topic}")
        # Get the top 3 words for this topic
        if assigned_topic != -1:
            topic_words = topic_model.get_topic(assigned_topic)
            top_3_words = [word for word, _ in topic_words[:3]]
            ret.append(top_3_words)
            #print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
        else:
            outlier_count += 1
            ret.append([])
    #print (f"number of outliers is {outlier_count}")
    return ret
sents = [d["text"] for d in data2]
rets = get_topics(sents)

In [22]:
# train_data = []
# test_data = []

def assign_topics(dataset):
    docs = [d["text"] for d in dataset]
    topic_words = get_topics(docs)
    test_data = []
    count = 0
    for i, data in enumerate(dataset):
        txt = data["text"]
        new_txt = txt
        words = topic_words[i]
        if len(words) != 0:
            new_txt = txt + ". Topics: " + ", ".join(words)
        else:
            count += 1
        test_data.append({"text":  new_txt, "labels": data["labels"]})
    print (f"lenght of data is {len(test_data)}")
    print (f"number of input with no topics:  {count}")
    return test_data
    
test_data = assign_topics(new_data1)
train_data = assign_topics(new_data2)


lenght of data is 230
number of input with no topics:  79
lenght of data is 914
number of input with no topics:  353


In [23]:
with open("stratified_train_data_topic.json", "w") as f:
    json.dump(train_data, f, indent=4)

with open("stratified_test_data_topic.json", "w") as f:
    json.dump(test_data, f, indent=4)